# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RajatBharti11/Rajat/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

1.  **What one row means for your lane:** Each row represents a unique customer's subscription status at the end of a given month.
2.  **Which table(s) you'll use:** I will use the `customer_subscriptions` table and potentially `customer_usage_data` for feature engineering.
3.  **Which time window:** I will analyze data for a full calendar year, e.g., from `2025-01-01` to `2025-12-31`, with a prediction window for the subsequent month.
4.  **What you'd predict or rank (label or proxy):** I will predict `churn`, a binary label (1 if the customer cancels their subscription in the following month, 0 otherwise).
5.  **One thing you deliberately exclude:** I will exclude personally identifiable information (PII) such as email addresses or names, as they are not relevant for predicting churn and introduce privacy risks.

In [9]:
import pandas as pd
import numpy as np
# For BigQuery integration (if using real data)
# import pandas_gbq

# Placeholder for your GCP Project ID
# BQ_PROJECT = 'your-gcp-project-id' # Replace with your actual project ID

# --- Simulate Data for Demonstration (if not connecting to BQ) ---
# If you have actual BigQuery tables, replace this simulation with pandas_gbq.read_gbq calls.
def simulate_customer_data(num_customers=1000, start_date='2025-01-01', end_date='2026-06-30'):
    np.random.seed(42)
    dates = pd.to_datetime(pd.date_range(start=start_date, end=end_date, freq='M'))

    data = []
    for i in range(num_customers):
        customer_id = f'cust_{i+1}'
        sub_start = pd.to_datetime(np.random.choice(pd.date_range(start='2024-01-01', end='2025-10-31', freq='D')))
        monthly_bill = np.random.uniform(20, 150)
        payment_method = np.random.choice(['Credit Card', 'PayPal', 'Direct Debit'])
        has_promo = np.random.choice([True, False], p=[0.2, 0.8])

        for month_end in dates:
            # Simulate active status for the month
            is_active = (month_end >= sub_start)

            # Simulate churn for the next month
            # Churn probability is higher for customers with lower tenure, higher bill, or 'PayPal'
            churn_prob = 0.05
            if (month_end - sub_start).days / 30 < 6: # New customers
                churn_prob += 0.05
            if monthly_bill > 100: # High bill
                churn_prob += 0.02
            if payment_method == 'PayPal': # Less sticky payment method
                churn_prob += 0.03

            # Make churn decision for the *next* month based on current month's status
            churn_next_month = 0
            if is_active and np.random.rand() < churn_prob: # Customer churns in the next month
                churn_next_month = 1

            # Simulate last activity date
            days_since_last_activity = np.random.randint(1, 60) if is_active else np.nan

            data.append({
                'customer_id': customer_id,
                'month_end': month_end,
                'subscription_start_date': sub_start,
                'monthly_bill': monthly_bill,
                'payment_method': payment_method,
                'has_promo_discount': has_promo,
                'is_active_this_month': is_active,
                'num_days_since_last_activity': days_since_last_activity,
                'churn_next_month': churn_next_month # This is our label
            })
    return pd.DataFrame(data)

# Generate simulated data
customer_subscriptions_df = simulate_customer_data()

print("Simulated customer_subscriptions_df created. Use this for demonstration.")
# If using BigQuery, you would replace the above with something like:
# query = """SELECT * FROM `your_gcp_project_id.your_dataset.customer_subscriptions`"""
# customer_subscriptions_df = pandas_gbq.read_gbq(query, project_id=BQ_PROJECT)


/tmp/ipykernel_1172/1683737402.py:13: FutureWarning: 'M' is deprecated and will be removed in a future version, please use 'ME' instead.
  dates = pd.to_datetime(pd.date_range(start=start_date, end=end_date, freq='M'))


Simulated customer_subscriptions_df created. Use this for demonstration.


In [10]:
# Display info and head of the simulated data
display(customer_subscriptions_df.info())
display(customer_subscriptions_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18000 entries, 0 to 17999
Data columns (total 9 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_id                   18000 non-null  object        
 1   month_end                     18000 non-null  datetime64[ns]
 2   subscription_start_date       18000 non-null  datetime64[ns]
 3   monthly_bill                  18000 non-null  float64       
 4   payment_method                18000 non-null  object        
 5   has_promo_discount            18000 non-null  bool          
 6   is_active_this_month          18000 non-null  bool          
 7   num_days_since_last_activity  16080 non-null  float64       
 8   churn_next_month              18000 non-null  int64         
dtypes: bool(2), datetime64[ns](2), float64(2), int64(1), object(2)
memory usage: 1019.7+ KB


None

,customer_id,month_end,subscription_start_date,monthly_bill,payment_method,has_promo_discount,is_active_this_month,num_days_since_last_activity,churn_next_month
0,cust_1,2025-01-31,2024-04-12,123.550588,Direct Debit,False,True,39.0,0
1,cust_1,2025-02-28,2024-04-12,123.550588,Direct Debit,False,True,23.0,0
2,cust_1,2025-03-31,2024-04-12,123.550588,Direct Debit,False,True,24.0,1
3,cust_1,2025-04-30,2024-04-12,123.550588,Direct Debit,False,True,40.0,0
4,cust_1,2025-05-31,2024-04-12,123.550588,Direct Debit,False,True,22.0,0


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Based on our churn prediction goal, here's how we'll classify the fields:

### Features (Knowable at decision moment)
*   `monthly_bill`: The recurring monthly charge for the subscription.
*   `tenure_months`: The duration (in months) the customer has been subscribed until the end of the current month.
*   `payment_method`: The method used for subscription payments (e.g., Credit Card, PayPal, Direct Debit).
*   `num_days_since_last_activity`: The number of days since the customer last interacted with the service.
*   `has_promo_discount`: A boolean indicating if the customer is currently on a promotional discount.

### Label (What we want to predict)
*   `churn_next_month`: A binary indicator (1 if the customer churns in the *next* month, 0 otherwise).

### Context (Useful for analysis but not directly for prediction in the model)
*   `customer_id`: Unique identifier for each customer.
*   `month_end`: The end date of the month for which the data is recorded.
*   `subscription_start_date`: The original start date of the customer's subscription.

### Excluded (Not used for prediction)
*   *Personally Identifiable Information (PII)*: As stated in Section 1, any PII (e.g., email, name) would be excluded for privacy reasons and irrelevance to the prediction task.

In [11]:
# Define the target prediction month
PREDICTION_MONTH_END = '2026-03-31'

# Filter data for the specific prediction month
feature_frame = customer_subscriptions_df[customer_subscriptions_df['month_end'] == PREDICTION_MONTH_END].copy()

# Feature Engineering
# 1. monthly_bill: Already available

# 2. tenure_months
# Knowable at the decision moment because it's calculated based on the subscription start date up to the current `month_end`.
feature_frame['tenure_months'] = ((feature_frame['month_end'] - feature_frame['subscription_start_date']).dt.days / 30).astype(int)

# 3. payment_method: Already available (needs encoding later)
# Knowable at the decision moment because it's a static attribute of the customer's subscription.

# 4. num_days_since_last_activity: Already available
# Knowable at the decision moment because it reflects recent customer interaction up to `month_end`.

# 5. has_promo_discount: Already available
# Knowable at the decision moment because it's a current status of the customer's subscription.

# Select the features and the label for the feature frame
final_features = [
    'customer_id',
    'month_end',
    'monthly_bill',
    'tenure_months',
    'payment_method',
    'num_days_since_last_activity',
    'has_promo_discount',
    'churn_next_month' # This is our label
]

feature_frame = feature_frame[final_features]

display(feature_frame.head())
display(feature_frame.info())

,customer_id,month_end,monthly_bill,tenure_months,payment_method,num_days_since_last_activity,has_promo_discount,churn_next_month
14,cust_1,2026-03-31,123.550588,23,Direct Debit,44.0,False,0
32,cust_2,2026-03-31,21.724445,21,Credit Card,57.0,False,0
50,cust_3,2026-03-31,56.521486,20,Credit Card,35.0,True,0
68,cust_4,2026-03-31,112.386176,14,PayPal,39.0,True,0
86,cust_5,2026-03-31,20.903777,18,Direct Debit,26.0,False,0


<class 'pandas.core.frame.DataFrame'>
Index: 1000 entries, 14 to 17996
Data columns (total 8 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_id                   1000 non-null   object        
 1   month_end                     1000 non-null   datetime64[ns]
 2   monthly_bill                  1000 non-null   float64       
 3   tenure_months                 1000 non-null   int64         
 4   payment_method                1000 non-null   object        
 5   num_days_since_last_activity  1000 non-null   float64       
 6   has_promo_discount            1000 non-null   bool          
 7   churn_next_month              1000 non-null   int64         
dtypes: bool(1), datetime64[ns](1), float64(2), int64(2), object(2)
memory usage: 63.5+ KB


None

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We will now verify three key aspects of our data slice for the month of `2026-03-31`:

1.  **Grain Verification:** Ensure that each row truly represents a unique customer's status at the end of the specified month.
2.  **Slice's Row Count and Date Span:** Confirm the total number of records and the exact date range covered by our chosen `PREDICTION_MONTH_END`.
3.  **Availability Check:** Verify the number of rows that survive after filtering for `is_active_this_month` (implicitly, customers for whom we can make a prediction).

## Leakage Experiment: The Trap

*Add one label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number — the leakage lesson from notebook 02, performed on real warehouse data by you.*

We will demonstrate data leakage by intentionally creating a feature that is directly derived from the label (`churn_next_month`). This will inflate our model's performance metrics, making it seem much better than it actually is.

For this example, let's create a synthetic leaky feature: `monthly_bill_next_month_decreased`. This feature would only be knowable *after* the churn event, but if we mistakenly include it, it will create leakage.

In [12]:
print("--- Leakage Experiment: Part 1 - Adding a Leaky Feature ---")

# Create a deliberately leaky feature:
# Imagine we have 'insight' into whether the monthly bill will decrease next month.
# In a real scenario, this 'insight' might actually be derived from observing churn.
# Here, we'll directly link it to churn for demonstration.
feature_frame_leaky = feature_frame.copy()
feature_frame_leaky['monthly_bill_next_month_decreased'] = feature_frame_leaky['churn_next_month'].apply(lambda x: 1 if x == 1 else 0) # High correlation with churn!

display(feature_frame_leaky.head())

--- Leakage Experiment: Part 1 - Adding a Leaky Feature ---


,customer_id,month_end,monthly_bill,tenure_months,payment_method,num_days_since_last_activity,has_promo_discount,churn_next_month,monthly_bill_next_month_decreased
14,cust_1,2026-03-31,123.550588,23,Direct Debit,44.0,False,0,0
32,cust_2,2026-03-31,21.724445,21,Credit Card,57.0,False,0,0
50,cust_3,2026-03-31,56.521486,20,Credit Card,35.0,True,0,0
68,cust_4,2026-03-31,112.386176,14,PayPal,39.0,True,0,0
86,cust_5,2026-03-31,20.903777,18,Direct Debit,26.0,False,0,0


In [13]:
print("--- Leakage Experiment: Part 2 - Simulating Model Training with Leaky Feature ---")

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Prepare data for a simple model
X_leaky = feature_frame_leaky.drop(columns=['customer_id', 'month_end', 'churn_next_month'])
y_leaky = feature_frame_leaky['churn_next_month']

# Define categorical and numerical features
categorical_features = ['payment_method']
numerical_features = ['monthly_bill', 'tenure_months', 'num_days_since_last_activity', 'has_promo_discount', 'monthly_bill_next_month_decreased']

# Create a column transformer for preprocessing
preprocessor_leaky = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ])

# Create a pipeline with preprocessing and logistic regression
model_leaky = Pipeline(steps=[('preprocessor', preprocessor_leaky),
                                ('classifier', LogisticRegression(solver='liblinear', random_state=42))])

# Split data (even though it's a single month, we'll simulate a split for evaluation)
X_train_leaky, X_test_leaky, y_train_leaky, y_test_leaky = train_test_split(X_leaky, y_leaky, test_size=0.3, random_state=42, stratify=y_leaky)

# Train the model
model_leaky.fit(X_train_leaky, y_train_leaky)

# Make predictions
y_pred_leaky = model_leaky.predict(X_test_leaky)

# Evaluate the model with leakage
print(f"\nAccuracy with leaky feature: {accuracy_score(y_test_leaky, y_pred_leaky):.4f}")
print("Classification Report with leaky feature:\n", classification_report(y_test_leaky, y_pred_leaky))

--- Leakage Experiment: Part 2 - Simulating Model Training with Leaky Feature ---

Accuracy with leaky feature: 1.0000
Classification Report with leaky feature:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00       280
           1       1.00      1.00      1.00        20

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300



In [14]:
print("--- Leakage Experiment: Part 3 - Removing Leaky Feature and Re-evaluating ---")

# Remove the leaky feature
X_honest = feature_frame.drop(columns=['customer_id', 'month_end', 'churn_next_month'])
y_honest = feature_frame['churn_next_month']

# Define categorical and numerical features (excluding the leaky one)
categorical_features_honest = ['payment_method']
numerical_features_honest = ['monthly_bill', 'tenure_months', 'num_days_since_last_activity', 'has_promo_discount']

# Create a column transformer for preprocessing without the leaky feature
preprocessor_honest = ColumnTransformer(
    transformers=[
        ('num', 'passthrough', numerical_features_honest),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features_honest)
    ])

# Create a pipeline without the leaky feature
model_honest = Pipeline(steps=[('preprocessor', preprocessor_honest),
                                 ('classifier', LogisticRegression(solver='liblinear', random_state=42))])

# Split data again
X_train_honest, X_test_honest, y_train_honest, y_test_honest = train_test_split(X_honest, y_honest, test_size=0.3, random_state=42, stratify=y_honest)

# Train the honest model
model_honest.fit(X_train_honest, y_train_honest)

# Make predictions
y_pred_honest = model_honest.predict(X_test_honest)

# Evaluate the honest model
print(f"\nAccuracy without leaky feature: {accuracy_score(y_test_honest, y_pred_honest):.4f}")
print("Classification Report without leaky feature:\n", classification_report(y_test_honest, y_pred_honest))

--- Leakage Experiment: Part 3 - Removing Leaky Feature and Re-evaluating ---

Accuracy without leaky feature: 0.9333
Classification Report without leaky feature:
               precision    recall  f1-score   support

           0       0.93      1.00      0.97       280
           1       0.00      0.00      0.00        20

    accuracy                           0.93       300
   macro avg       0.47      0.50      0.48       300
weighted avg       0.87      0.93      0.90       300



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [15]:
print(f"--- Verification Queries for Month: {PREDICTION_MONTH_END} ---")

# Query 1: Grain Verification (One row = one customer at month end)
# We expect the number of unique (customer_id, month_end) pairs to equal the total number of rows.
# This confirms the unit of analysis: a customer's status at a specific month's end.
num_unique_grains = feature_frame.drop_duplicates(subset=['customer_id', 'month_end']).shape[0]
total_rows = feature_frame.shape[0]

print(f"\n1. Grain Verification:")
print(f"   Number of unique (customer_id, month_end) combinations: {num_unique_grains}")
print(f"   Total rows in feature frame: {total_rows}")
print(f"   Grain is unique: {num_unique_grains == total_rows}")

# Query 2: Slice's Row Count and Date Span
print(f"\n2. Slice's Row Count and Date Span:")
print(f"   Row count for {PREDICTION_MONTH_END}: {total_rows}")
min_date = feature_frame['month_end'].min()
max_date = feature_frame['month_end'].max()
print(f"   Date span (min_month_end to max_month_end): {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
print(f"   Confirmed single month: {min_date == pd.to_datetime(PREDICTION_MONTH_END) and max_date == pd.to_datetime(PREDICTION_MONTH_END)}")

# Query 3: Availability (Filter for active customers)
# We assume 'is_active_this_month' is available from the original data (or can be derived)
# and represents customers for whom a prediction is meaningful.
active_customers_count = customer_subscriptions_df[(customer_subscriptions_df['month_end'] == PREDICTION_MONTH_END) & (customer_subscriptions_df['is_active_this_month'] == True)].shape[0]

print(f"\n3. Availability Check (IS TRUE):")
print(f"   Total rows for {PREDICTION_MONTH_END}: {total_rows}")
print(f"   Rows where 'is_active_this_month' is TRUE: {active_customers_count}")
print(f"   Percentage of active customers: {active_customers_count / total_rows:.2%}")

--- Verification Queries for Month: 2026-03-31 ---

1. Grain Verification:
   Number of unique (customer_id, month_end) combinations: 1000
   Total rows in feature frame: 1000
   Grain is unique: True

2. Slice's Row Count and Date Span:
   Row count for 2026-03-31: 1000
   Date span (min_month_end to max_month_end): 2026-03-31 to 2026-03-31
   Confirmed single month: True

3. Availability Check (IS TRUE):
   Total rows for 2026-03-31: 1000
   Rows where 'is_active_this_month' is TRUE: 1000
   Percentage of active customers: 100.00%


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

While our current data provides valuable insights into customer churn, there are inherent limitations that need to be acknowledged:

1.  **Limited Historical Context:** We are using a single month's snapshot (`2026-03-31`) to build features. While we calculate `tenure_months`, more granular historical customer behavior (e.g., changes in usage patterns, past billing issues, multiple past churn attempts) is not captured in this feature frame. This limits our ability to identify complex, time-dependent churn drivers.
2.  **Imbalance in Churn Rates:** Churn events are often rare compared to customer retention. Our simulated data will likely exhibit a class imbalance between churned and retained customers, which can affect model training and evaluation metrics if not handled appropriately.
3.  **Missing External Factors:** The dataset only includes internal customer and subscription data. External factors that could influence churn (e.g., competitor actions, economic downturns, news events, new product launches) are not included and cannot be inferred from this data alone.
4.  **No Direct Causality:** While we can build a predictive model, correlation does not imply causation. The model might identify strong predictors, but it won't directly tell us *why* a customer churns. Further qualitative research or experimentation would be needed to understand causal factors.
5.  **Small Window of Observation:** The prediction window is for the *next* month. We cannot predict churn further into the future with this setup, nor can we capture very slow, long-term churn processes that might unfold over many months.

In [16]:
print("--- Data Limits: Illustrating Class Imbalance ---")

# Check the distribution of the churn label in our feature frame
churn_distribution = feature_frame['churn_next_month'].value_counts(normalize=True)

print(f"\nDistribution of 'churn_next_month' for {PREDICTION_MONTH_END}:")
print(churn_distribution)

if churn_distribution.min() < 0.2:
    print("\nObservation: The churn class is significantly imbalanced, which is a common data limitation in churn prediction. This could lead to models that perform well on the majority class (retained customers) but poorly on the minority class (churned customers).")
else:
    print("\nObservation: The churn class is relatively balanced in this simulated data, which is less common in real-world scenarios but demonstrates the concept of checking for imbalance.")

# A limitation: this particular slice (single month) might not fully represent overall churn dynamics.

--- Data Limits: Illustrating Class Imbalance ---

Distribution of 'churn_next_month' for 2026-03-31:
churn_next_month
0    0.932
1    0.068
Name: proportion, dtype: float64

Observation: The churn class is significantly imbalanced, which is a common data limitation in churn prediction. This could lead to models that perform well on the majority class (retained customers) but poorly on the minority class (churned customers).


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.